# Static-grid Kirchhoff–Carrier string

Define the ranges **before** synthesising or fitting. Both sounds and all optimiser
evaluations use the same grid and time step. Run all cells, then use the buttons.

The controls are taut-string fundamental $f_0$, inharmonicity $B$, amplitude $a$,
radius $r$, and losses $\sigma_0,\sigma_1$. Density and length are fixed by hand.
The stiff, small-amplitude first mode is $f_1=f_0\sqrt{1+B}$, not $f_0$ itself.

$$T_0=4\rho\pi r^2 L^2 f_0^2,\qquad
E=\frac{BT_0L^2}{\pi^2 I},\qquad I=\frac{\pi r^4}{4}.$$

The native non-SAV update and reference startup convention are preserved. The
physical and calculus derivations remain in [the teaching notebook](kc_string.ipynb).
The controls here call shared modules; they do not redefine the simulator.

## 1. Ranges and a shared numerical configuration

We preserve the nominal reference string's spatial resolution at 44.1 kHz and ×1
(148 intervals). Starting at ×1, choose the smallest integer oversampling factor
that supports at least this many intervals at the box's worst corner. At that
factor, take the largest stable even interval count, using Russo's 1% spacing
margin. The defaults give ×3 and 176 intervals.

With $H=(L/N)/1.01$ and $k=1/(f_s\,OF)$, the checked condition is
$$\frac{c^2k^2+4\sigma_1k}{H^2}+\frac{4K^2k^2}{H^4}\leq1.$$
The worst corner uses maximum $f_0,B,\sigma_1$. The bending term has the fourth
power of the spacing margin; a uniform 0.980 score is not the exact same margin.
Unlike MATLAB's reference-comparison path, this workflow keeps the internal rate
at an exact integer multiple of the output rate, without a subsequent adjustment.

**Stability is not an accuracy guarantee.** Modal detuning is reported for the
linear damped model; it does not prove accuracy of nonlinear pitch glides.
No dispersion threshold, adaptive refinement, spectral mask or staged duration
is used. The nonlinear-strength indicator $B(a/r)^2$ describes a first-mode start;
it is not an accuracy bound for the multimode or raised-cosine excitation.

Edit ranges and click **Apply ranges / prepare grid**. Editing configuration again
invalidates audio and fits. Moving sliders inside the prepared box does not alter
the grid. Resource limits reject expensive configurations rather than coarsening.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import torch
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/kc_string.py').exists())
if str(root) not in sys.path: sys.path.insert(0,str(root))
from src.optimization_workbench import OptimizationWorkbench
torch.set_num_threads(1)
workbench = OptimizationWorkbench()


In [ ]:
display(workbench.ranges_panel())

## 2. Choose and synthesise the target

Eight modes with weights $1/n^2$ are the default. The midpoint listener suppresses even spatial modes. Playback uses a shared gain; fitting uses the original displacement signal.

In [ ]:
display(workbench.target_panel())
for checkbox in workbench.optimize_boxes: checkbox.layout.display="none"

## Result

This notebook is synthesis-only: do not interpret its parameter-selection checkboxes as an optimisation run. They are shared controls for the companion workbench. `workbench.target` holds the audio and parameters; `workbench.grid` holds the immutable discretisation. The table reports numerical detuning, including the linear damping discretisation. See [the optimisation notebook](optimization.ipynb) to fit a target.